In [ ]:
import os 
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell  
from ipywidgets import interact  
InteractiveShell.ast_node_interactivity = "all"   # type: ignore
import ipywidgets as widgets
from ipywidgets import interact 
from pandas import Index
from typing import Any, TypedDict

In [ ]:
%matplotlib notebook

In [ ]:
def show_col_summary(column: Any, frame):
    print(frame[column].describe())

def show_summary(dfs: list[pd.DataFrame], v_dir: Path):
    print(f"Results aggregated over {v_dir.stem} in {v_dir.parent.parent.stem}/{v_dir.parent.stem}")
    for frame in dfs:
        interact(show_col_summary, frame=frame, column=list(frame.columns))
        frame 


In [ ]:
def all_corr_store_metrics(dir: Path) -> pd.DataFrame:
    metrics = sorted(dir.rglob("*", case_sensitive=False))
    metrics_dfs = map(lambda m: pd.read_csv(m), metrics) 
    run_metrics = pd.concat(metrics_dfs, axis=0)
    return run_metrics

def viz_gcs_metrics(v_dir: Path) -> None:
    run_dirs = sorted(v_dir.glob("run*", case_sensitive=False))
    for r_dir in run_dirs:
        gcs_metrics_dirs = sorted(r_dir.glob("train/GCS*/", case_sensitive=False))
        rgcs_metrics_dirs = sorted(r_dir.glob("train/RGCS*/", case_sensitive=False))
        # Concat metrics into a single stitch to store for train/gch_metrics.csv
        for g_dir, rg_dir in zip(gcs_metrics_dirs, rgcs_metrics_dirs):
            run_gcs_df = all_corr_store_metrics(g_dir)
            run_rgcs_df = all_corr_store_metrics(rg_dir)
            run_gcs_df.to_csv(g_dir.parent.resolve() / "run_gcs.csv", index=False)
            run_rgcs_df.to_csv(rg_dir.parent.resolve() / "run_rgcs.csv", index=False)

    # Within 1 version perform summary stats 
    # TODO: Check that bug is not here, where we only get file for current run, vs parent
    run_metrics_dirs = sorted(v_dir.rglob("run_rgcs.csv", case_sensitive=False))
    viz_standard_metrics(run_metrics_dirs, "rgcs")
    
    run_metrics_dirs = sorted(v_dir.rglob("run_gcs.csv", case_sensitive=False))
    # TODO: Remove pass of name and instead get it from file names
    viz_standard_metrics(run_metrics_dirs, "gcs")

In [ ]:
class SummaryMetricInfo(TypedDict):
    values: np.ndarray
    save_path: Path
    header: list[str]

In [ ]:
def viz_store_sep_col(metric: SummaryMetricInfo, save_fig: bool = True, save_derived_metric: bool = False) -> None:
    """
    For a metric object instance, visualize each col of `values` with its associated string in `header`.
    Internal output is figure with plots per column, header label combo.
    Optionally save each to compressed numpy file with flags (NOT recommended).
    Optionally save figure (Recommended). 
    TODO: But with what path, might as well maybe create separate instance separate from global df, not really. 
    """
    n_cols = metric["values"].shape[1]
    fig, axs = plt.subplots(1, n_cols, sharex=True, sharey=True, figsize=(4 * n_cols, 4))
    fig.subplots_adjust(wspace=0, hspace=0)

    for i, _ in enumerate(metric["header"]):
        col_vals = metric["values"][:,i]
        col_name = metric["header"][i]
        col_path = Path(metric['save_path'].resolve() / metric["header"][i])
        axs[i].plot(
            col_vals, 
            linestyle="dashed", 
            color="black", 
            marker="o", 
            label=col_name
        )
        axs[i].legend()

        if save_derived_metric:
            np.savez_compressed(file=col_path, args=col_vals)

    if save_fig: 
        fig.savefig(metric['save_path'].resolve(), format="png")

def viz_store_all_col(metric: SummaryMetricInfo, save_fig: bool = True, save_derived_metric: bool = True) -> None:
    """
    For a metric object instance, visualize all col of `values` with its associated string in `header`.
    Internal output is figure with 1 plot for all column , labelled by col name in Index.
    Optionally save each to compressed numpy file with flags (Recommended).
    Optionally save figure (Recommended). 
    """
    fig, axs = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(4, 4))
    metric_df = pd.DataFrame(metric["values"], columns=metric['header'])
    metric_df.plot(sharex=True, sharey=True, ax=axs)

    if save_derived_metric:
        metric_df.to_csv(metric['save_path'].resolve(), index=False)

    if save_fig: 
        fig.savefig(metric['save_path'].resolve(), format="png")

# TODO: Make save_config options for metric instance (global and for_each, is this coupling best?)
# TODO: Best way even is custom metric objects per derived column and setting it there maybe
def viz_store_summary_metrics(summary_metrics: list[SummaryMetricInfo], save_fig: bool = True, save_derived_metric: bool = False) -> None:
    """ 
    SummaryMetricInfo instances in list are plotted along figures.
    TODO: This can be decoupled for GCS vs standard metric, do not need to force these to come one after the other. 
    Unclear what aggegration styles occur. 
    """
    for i, summary_metric in enumerate(summary_metrics):
        print(f"Summary metric immediate file results and subdirs all saved into {summary_metric['save_path'].resolve()}")
        viz_store_sep_col(summary_metric)
        viz_store_all_col(summary_metric)

In [ ]:
def viz_standard_metrics(metrics_paths: list[Path], prefix:str) -> None:
    num_metrics = len(metrics_paths)

    # Sizes are unknown unless peeking is performed
    sum = np.zeros(0)
    sum_sq = np.zeros(0)
    header = [""]
    # Save initial path
    enumerated_metrics = enumerate(metrics_paths)

    for i, train_metrics_path in enumerated_metrics:
        with open(train_metrics_path) as f:
            cols = f.readline().split(',')
            ncols = len(cols)
            
        data = pd.read_csv(train_metrics_path.resolve(), usecols=range(0,ncols)).dropna()
        data = data.drop(columns=["step"])
        metrics = np.asarray(data.values)
        
        if i == 0:
            sum = np.zeros_like(metrics)
            sum_sq = np.zeros_like(metrics)
            header = list(data.columns)

        sum = np.sum([sum, metrics], axis=(0))
        sum_sq = np.sum([sum_sq, np.square(metrics)], axis=(0))

    global_mean = sum / num_metrics
    global_var = (sum_sq / num_metrics) - (np.square(global_mean))
    global_std = np.sqrt(global_var)
    dir = list(metrics_paths)[0].parent
    summary_metrics = [
        SummaryMetricInfo(
            values=global_std,
            save_path=dir / "global_std",
            header=["global_std" + h for h in header]
        ),
        SummaryMetricInfo(
            values=global_mean,
            save_path= dir/ "global_std",
            header=["global_mean" + h for h in header]
        ),
    ]
    viz_store_summary_metrics(summary_metrics)

In [ ]:
outputs_dir = Path().resolve().parent / "outputs"
plt.switch_backend('WebAgg')

for experim_path in sorted(outputs_dir.iterdir()):
    version_dirs = sorted(experim_path.glob("multijob/v*"))
    for v_dir in version_dirs:
        train_metrics_paths = sorted(v_dir.rglob("run*/train/metrics.csv", case_sensitive=False))
        viz_standard_metrics(train_metrics_paths, "standard")
        viz_gcs_metrics(v_dir)

plt.show()